In [1]:

import pandas as pd
import pickle

from typing import List
from fastf1.core import Session

import shared_libraries.data_processing as processing

import matplotlib.pyplot as plt
import os

In [2]:
with open("./sessions.pickle", "rb") as file:
    sessions: List[Session] = pickle.load(file)

In [3]:
compound_map = pd.read_csv("./compounds_map.csv")

First, let's get our lap data for all sessions.

In [4]:
data_by_circuit = processing.get_data_by_circuit(sessions, compound_map)


Next, we remove laps during which special compounds were used, as compounds other than *SOFT*, *MEDIUM* and *HARD* are outside the scope of this experiment.

Also, we remove laps during which track status was not *green* or *yellow*. That way, we get rid of laps affected by unexpected events. These laps introduce noise and anomalies into the data, which is why we remove them.

In [5]:
for circuit, df in data_by_circuit.items():
    processing.remove_special_compounds(df)
    processing.remove_laps_affected_by_unexpected_events(df)


## Outliers
Below, we look at lap time distributions and outliers for each circuit. Figures are saved to the "figures/" folder.

Before analysis, laps are split into three groups:
1. First laps — first laps take longer due to stationary start.
1. Pit laps — take longer due to pit stop.
1. Remaining laps.

Outliers will be removed afterwards, respecting the aforementioned data split. Treating the three groups of laps separately avoids issues such as first laps or pit laps being treated as outliers (which they often are relative to the remaining, faster laps).

Also, any first laps with pit stop are removed. Taking a pit stop on the first lap is not part of normal racing strategy and creates large outliers.

In [6]:
data_split = {}
for circuit, df in data_by_circuit.items():
    processing.remove_first_laps_with_pit_stop(df)
    data_split[circuit] = {}
    first_laps = df.drop(df[df["LapNumber"] != 1].index)
    other_laps = df.drop(df[df["LapNumber"] == 1].index)
    pit_laps = df.drop(df[~df["IsPitLap"]].index)
    non_pit_laps = df.drop(df[df["IsPitLap"]].index)

    data_split[circuit]["first_laps"] = first_laps
    data_split[circuit]["pit_laps"] = pit_laps
    data_split[circuit]["non_pit_laps"] = non_pit_laps


In [7]:
# Creating and saving figures
for circuit, df in data_by_circuit.items():

    path = f"./figures/lap_times"
    os.makedirs(path, exist_ok=True)

    figures_and_axes = {}

    for part_key in data_split[circuit]:
        figures_and_axes[part_key] = {}
        figures_and_axes[part_key]["histogram"] = plt.subplots()
        figures_and_axes[part_key]["box_plot"] = plt.subplots()

    for part_key, plot_type_and_figs in figures_and_axes.items():
        for plot_type, (fig, ax) in plot_type_and_figs.items():
            if plot_type == "histogram":
                ax.set_xlabel("Lap time (s)")
                ax.hist(data_split[circuit][part_key]["LapTimeSeconds"])
            elif plot_type == "box_plot":
                ax.set_ylabel("Lap time (s)")
                ax.boxplot(data_split[circuit][part_key]["LapTimeSeconds"])
            
            if part_key == "non_pit_laps":
                title = "Laps without pit stop"
            elif part_key == "pit_laps":
                title = "Pit laps"
            else:
                title = "First laps"
            ax.set_title(f"{title} — {circuit}")
            fig.savefig(f"{path}/{circuit}_{part_key}_{plot_type}")
            plt.close(fig)


In [8]:


# Removing outliers
for circuit, df in data_by_circuit.items():
    data_parts = []
    for data_part in data_split[circuit].values():
        processing.remove_outliers(data_part)
        data_parts.append(data_part)

    data_by_circuit[circuit] = pd.concat(data_parts, axis="index", ignore_index=True)



## Preparing data for ML
In this stage, we process the data further to make it suitable for regression tasks. The following operations are performed:
1. Removal of missing values.
1. Addition of a "RealCompound" column which shows what compound was used during a lap. This is necessary because for every session, different compounds (*C1*, *C2*, ..., *C5*) are chosen for *SOFT*, *MEDIUM* and *HARD* tyres.
1. Conversion of numeric WindDirection values to categorical values such as *N*, *S* or *NW*.
1. Removal of columns that were not selected as features for regression.
1. One-hot encoding of categorical columns.

In [9]:
# Preparing data for ML — no one-hot encoding yet
for circuit, df in data_by_circuit.items():
    processing.remove_missing_values(df)
    df.reset_index(inplace=True)
    df = df.convert_dtypes()
    processing.add_real_compound(df)
    processing.make_wind_direction_categorical(df)
    processing.select_columns_for_ml(df)
    df = pd.get_dummies(df)
    processing.add_missing_dummy_columns(df)
    processing.sort_columns(df)
    data_by_circuit[circuit] = df

Save clean data without one-hot encoding

In [10]:
os.makedirs("./ig_data", exist_ok=True)
with open("ig_data/data_by_circuit.pickle", "wb") as file:
    pickle.dump(data_by_circuit, file)

In [11]:
for circuit, df in data_by_circuit.items():
    df = pd.get_dummies(df)
    processing.add_missing_dummy_columns(df)
    processing.sort_columns(df)
    
    data_by_circuit[circuit] = df

Save data with one-hot encoding

In [12]:
with open("ig_data/data_by_circuit_one_hot_encoded.pickle", "wb") as file:
    pickle.dump(data_by_circuit, file)

In [13]:
circuits = []
circuit_lap_counts = []
for circuit, df in data_by_circuit.items():
    lap_count = df["LapNumber"].max()
    circuits.append(circuit)
    circuit_lap_counts.append(lap_count)

pd.DataFrame({
    "LapCount": circuit_lap_counts,
}, index=circuits).to_csv("./ig_data/circuit_lap_counts.csv")